# Dev-set error analysis (issue #25)

Confusion matrix + per-axis error breakdown on the committed dev
baseline (`eval_runs/dev_baseline.json`, composite **87.95**).

The computation lives in `scripts/error_analysis.py` (pure-stdlib +
the repo's vendor loader, no LLM/network; mirrors
`evaluation/scoring.py` so every count reconciles with the composite).
This notebook is a thin presentation layer — re-run top-to-bottom
after any new eval. The committed snapshot is
`eval_runs/error_analysis.json`; the narrative summary is design-doc
§9.

In [ ]:
import sys, json
from pathlib import Path

ROOT = Path.cwd().parents[0] if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))
from scripts import error_analysis as ea

preds, gt = ea.load(ea.DEFAULT_PREDICTIONS, ea.DEFAULT_GT)
print(len(preds), 'predictions /', len(gt), 'labelled rows')

## 1. Subcategory confusion matrix (top-15 worst true→predicted pairs)

In [ ]:
sc = ea.analyze_subcategory(preds, gt)
print(f"accuracy {sc['accuracy']:.1%}  ({sc['total_errors']} misses)\n")
for r in sc['worst_pairs']:
    print(f"  {r['true']:>22} -> {r['predicted']:<22} x{r['count']}")
print('\nhighest error-rate true subcategories:')
for r in sc['error_rate_by_true_subcategory'][:10]:
    print(f"  {r['true']:>22}  {r['errors']}/{r['n']}  ({r['error_rate']:.0%})")

## 2. Risk-level errors

In [ ]:
rk = ea.analyze_risk(preds, gt)
print(f"accuracy {rk['accuracy']:.1%}  ({rk['total_errors']} errors)")
print('over :', rk['over_predictions'])
print('under:', rk['under_predictions'])

## 3. HITL confusion matrix (TP / FP / FN / TN)

`hitl_f1` carries **15%** of the composite and is the lowest axis —
the biggest single point lever (→ #63 recall, #64 precision).

In [ ]:
h = ea.analyze_hitl(preds, gt)
print(f"             gt review   gt no-review")
print(f"pred review     TP={h['tp']:<4}      FP={h['fp']}")
print(f"pred no-rev     FN={h['fn']:<4}      TN={h['tn']}")
print(f"\nP={h['precision']:.3f}  R={h['recall']:.3f}  F1={h['f1']:.3f}")
print('FP by subcategory:', h['fp_by_subcategory'])
print('FN by subcategory:', h['fn_by_subcategory'])

## 4. Location-field mismatch (building / address / floor)

In [ ]:
loc = ea.analyze_location(preds, gt)
print(f"accuracy {loc['accuracy']:.1%}  ({loc['total_errors']} errors)")
print(f"  building={loc['building_misses']}  address={loc['address_misses']}  floor={loc['floor_misses']}")
print('  pattern:', loc['pattern'])

## 5. Vendor-match failure breakdown

Bucketed by first violated constraint. Note the
`emergency: all acceptable at_capacity` bucket is **AC-mandated by
issue #21** (skip at_capacity on emergencies) — a documented policy
decision (→ #67), not a bug.

In [ ]:
ven = ea.analyze_vendor(preds, gt)
print(f"accuracy {ven['accuracy']:.1%}  ({ven['total_errors']} errors)\n")
for k, v in ven['failure_buckets'].items():
    print(f"  {v:>3}  {k}")

## 6. Auto-resolution misses

In [ ]:
au = ea.analyze_auto_resolution(preds, gt)
print(f"rate {au['rate']:.1%}  ({au['hits']}/{au['eligible']} eligible)")
print('miss causes:', au['miss_by_cause'])

## Top 5 findings + refresh the committed artifact

Filed as follow-up issues **#63–#67** (see design-doc §9). The cell
below regenerates `eval_runs/error_analysis.json` so a notebook re-run
and the committed snapshot never drift.

In [ ]:
ea.print_report(sc, rk, h, loc, ven, au)
ea.save_json(sc, rk, h, loc, ven, au, ROOT / 'eval_runs' / 'error_analysis.json')